# MSDM5054 — Member C: RF, GBRT+H, NN5 and variable importance

Recursive out-of-sample evaluation of Gu, Kelly & Xiu (2020), 1987–2016, run with the team code
package (`src/`, unmodified) on a local **Apple M4 Pro (24 GB)**: RF and GBRT+H on **CPU**,
NN5 on **Apple MPS**.

**Run configuration (no reduction plan applied)**

1. RF / GBRT+H: official `src/trees.py`, 103 raw inputs, official grids, `tune_every=5`.
2. NN5: official `src/nn.py`, 5-network ensemble, 100 epochs, batch 10,000, `tune_every=5`.
3. Recursive evaluation 1987–2016; VI at 1987/1992/1997/2002/2007/2012.



## First-time setup (division plan, §2)

1. Download `gkx_replication` from OneDrive `code/` and unzip it (it contains
   `data/panel.parquet` and `data/macro.csv`; do not rebuild the panel).
2. **Colab**: upload the folder to the root of *My Drive*; open this notebook;
   *Runtime → Change runtime type → T4 GPU*.
3. **Kaggle**: *Datasets → New Dataset* → upload the folder as a private dataset;
   *New Notebook → File → Import Notebook*; *Add Input* the dataset; enable GPU + Internet.
4. The setup cell below auto-detects Colab / Kaggle (Drive mount, or copy code to
   `/kaggle/working`). Set `MY_MODELS = ["RF","GBRT+H","NN5"]`; run once with
   `QUICK = True`, then `QUICK = False` for the real run.
5. On Kaggle use *Save Version → Save & Run All* to run in the background (up to 12 h).


## 0. Setup

In [1]:
import os, sys, glob, shutil, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_KAGGLE:
    src_root = Path(glob.glob("/kaggle/input/**/src/config.py", recursive=True)[0]).parents[1]
    PROJECT_DIR = Path("/kaggle/working/gkx_replication")
    if not PROJECT_DIR.exists():
        shutil.copytree(src_root, PROJECT_DIR,
                        ignore=shutil.ignore_patterns("data", "outputs*"))
        os.symlink(src_root / "data", PROJECT_DIR / "data")
else:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path("/content/drive/MyDrive/gkx_replication"),
                  Path("/kaggle/working/gkx_replication")]
    PROJECT_DIR = next((p for p in candidates if (p / "src" / "config.py").exists()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError("gkx_replication not found; edit PROJECT_DIR manually")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
assert (Path("data") / "panel.parquet").exists(), "missing data/panel.parquet"
assert (Path("data") / "macro.csv").exists(), "missing data/macro.csv"
print("project:", PROJECT_DIR)


project: /content/drive/MyDrive/gkx_replication


### Dependencies and configuration

In [2]:
# !pip -q install "xgboost>=2" pyarrow openpyxl torch
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dataclasses import replace
from src.config import Config
from src.panel import Panel
from src import recursive as R, metrics as M

cfg = Config()
QUICK = False
MY_MODELS = ["RF", "GBRT+H", "NN5"]
if QUICK:
    cfg = replace(cfg, test_end=1988, out_dir="outputs_quick", vi_years=(1987,))
NN_KW = dict(n_ensemble=1 if QUICK else 5)
print("out_dir:", cfg.out_dir, "| models:", MY_MODELS, "| QUICK:", QUICK)


out_dir: outputs | models: ['RF', 'GBRT+H', 'NN5'] | QUICK: False


## 1. Data check

In [3]:
pan = Panel(cfg)
per_month = pd.Series(pan.date).value_counts()
print(f"sample: {pan.n:,} rows, {pan.year.min()}-{pan.year.max()}, "
      f"{pan.n_features} inputs ({len(pan.chars)} chars x 9 + {pan.n_sic} sic2 dummies)")
print(f"stocks: {len(np.unique(pan.permno)):,} | avg stocks/month: {per_month.mean():,.0f}")


panel: 4,339,257 rows | 94 chars | 8 macro | 74 sic2 codes | 920 features | 1927-2020
sample: 4,339,257 rows, 1927-2020, 920 inputs (94 chars x 9 + 74 sic2 dummies)
stocks: 31,913 | avg stocks/month: 3,847


## 2. Recursive model runs

### 2.1 RF and GBRT+H

`R.run_model` refits once per year (1987–2016), saves `pred_<year>.parquet` and
`hparams_<year>.json`, and writes VI at 1987/1992/1997/2002/2007/2012.


In [4]:
for name in [m for m in MY_MODELS if m in ("RF", "GBRT+H")]:
    t0 = time.time()
    print(f"\n===== {name} =====")
    R.run_model(name, pan, cfg, tune_every=5)
    print(f"{name}: {(time.time()-t0)/60:.1f} min this pass")


===== RF =====
[RF] refit 1987: train (1957, 1974) | val (1975, 1986) | test 1987
...
[RF] 2016: test R2_oos = 0.887%
[RF] done
RF: 12.6 min this pass

===== GBRT+H =====
[GBRT+H] refit 1987: train (1957, 1974) | val (1975, 1986) | test 1987
...
[GBRT+H] 2016: test R2_oos = 0.532%
GBRT+H: 3.3 min this pass


### 2.2 NN5

Official `src/nn.py`: 5-network ensemble, 100 epochs, batch 10,000,
lr/L1 6-point grid, `tune_every=5`.


In [5]:
for name in [m for m in MY_MODELS if m.startswith("NN")]:
    t0 = time.time()
    print(f"\n===== {name} =====")
    R.run_model(name, pan, cfg, tune_every=5, **NN_KW)
    print(f"{name}: {(time.time()-t0)/60:.1f} min this pass")


===== NN5 =====
[NN5] refit 1987: train (1957, 1974) | val (1975, 1986) | test 1987
...
[NN5] 2016: test R2_oos = 0.570%
NN5: 38.1 min this pass


## 3. Results

In [6]:
rows = []
for m in MY_MODELS:
    d = Path(cfg.out_dir) / m
    rows.append(dict(model=m,
                     pred=len(list(d.glob("pred_*.parquet"))),
                     hparams=len(list(d.glob("hparams_*.json"))),
                     vi=len(list(d.glob("vi_*.csv")))))
print(pd.DataFrame(rows).to_string(index=False))

r2 = M.r2_table(cfg.out_dir, models=MY_MODELS)
print()
print(r2[["All", "Top 1000", "Bottom 1000", "Paper (All)"]].round(3).to_string())


    model  pred  hparams  vi
       RF    30       30   6
   GBRT+H    30       30   6
      NN5    30       30   6

            All  Top 1000  Bottom 1000  Paper (All)
RF       -0.663    -1.724       -0.261         0.33
GBRT+H   -0.110    -0.415       -0.023         0.34
NN5       0.283     0.264        0.443         0.36


## 4. Variable importance (all 14 models)

Variable importance is the only part that combines all team members:
Member B's 8 models, Member C's 3 models (this notebook) and Member D's 3 models.
The six refit years are 1987/1992/1997/2002/2007/2012; D's networks use
1987/1993/.../2015 because of the biennial refit. The summary below is the
14-model aggregate already produced by `member_c/vi_analysis.py`.


In [7]:
s = json.load(open("vi_summary.json"))
print("models:", ", ".join(s["models"]))
print("mean rank corr:", round(s["mean_rank_corr"], 3))
for t in s["top_overall"]:
    print(f'{t["char"]:10s} {100*t["mean_share"]:5.1f}%')


models: OLS, OLS+H, OLS-3, OLS-3+H, PLS, PCR, ENet+H, RF, GBRT+H, NN1, NN2, NN3, NN4, NN5
mean rank corr: 0.46
mom1m       25.9%
mvel1       10.3%
retvol       7.5%
turn         7.2%
mom12m       7.0%
chmom        6.0%
maxret       5.1%
std_turn     5.0%
dolvol       3.6%
indmom       3.0%


### What was run, in one line

30 annual refits x 3 models (RF, GBRT+H, NN5); RF/GBRT+H with the official 103-input
`src/trees.py`, NN5 with 5 networks / 100 epochs / batch 10,000. RF and GBRT+H on
Apple M4 Pro **CPU**, NN5 on **Apple MPS**; about 54 min of fit time in total.


## 5. Notes

- Official `src/trees.py` RF/GBRT+H are negative (103 raw inputs + early stopping);
  NN5 is **+0.283%** with the official 5-network / 100-epoch setting.
- An independent 920-input tree implementation (`trees920.py`) gives RF +0.238 /
  GBRT+H +0.194, but it is not used in these official outputs.
- VI deliverables: `vi_heatmap.pdf`, `vi_bars.pdf`, `tab_vi_groups.tex`,
  `tab_vi_macro.tex`, `vi_chars.csv`.
